# Lab 4: Functions and Debugging with Hydrologic Indices

        **Week:** Week 4

        **Lab type:** Individual lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Use reusable functions.
- Read helper functions from a package.
- Calculate annual hydrologic indices.
- Practice debugging with small checks.

        ## Earth and environmental motivation

        Hydrologic indices such as annual mean flow, peak flow, and low flow become easier to check and reuse when they are written as functions.

        ## Dataset

        `data/processed/iowa_streamflow_daily.csv`

        ## Python concepts used

        - Functions
- Imports
- Return values
- Debugging
- Hydrologic indices

## Lab 3 Debrief and Collaborative Debugging (First 20 Minutes)

Open the debrief card from Lab 3. Two to four students or groups will
share a solved problem, an unresolved problem with evidence, or a verification
choice. The class will investigate one open problem one check at a time.

- 0-3 min: review the issue board.
- 3-11 min: student reports.
- 11-18 min: collaborative debugging.
- 18-20 min: record one reusable lesson and connect it to today's Lab.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

In [ ]:
import pandas as pd

from earthcourse.hydro import annual_mean_flow, annual_peak_flow, seven_day_low_flow

stream = pd.read_csv(PROCESSED_DIR / "iowa_streamflow_daily.csv", parse_dates=["date"])
print(stream.head())

In [ ]:
annual_mean = annual_mean_flow(stream)
low_flow = seven_day_low_flow(stream)
peak_flow = annual_peak_flow(stream)
indices = annual_mean.merge(low_flow, on="year").merge(peak_flow, on="year")
print(indices.head())

In [ ]:
assert (indices["mean_discharge_cfs"] > 0).all()
assert (indices["peak_discharge_cfs"] >= indices["low7_discharge_cfs"]).all()
print("Sanity checks passed.")

## Guided coding: write the function yourself first

Before trusting a library, write the simplest version yourself and test it on
data so small you can check the answer by hand. Here the 2020 mean must be
exactly (10 + 20) / 2 = 15.

In [ ]:
def annual_mean_flow_simple(df):
    """Return a table of annual mean discharge from daily data."""
    data = df.copy()
    data["year"] = data["date"].dt.year
    return data.groupby("year", as_index=False)["discharge_cfs"].mean()

tiny = pd.DataFrame({
    "date": pd.to_datetime(["2020-01-01", "2020-01-02", "2021-01-01"]),
    "discharge_cfs": [10.0, 20.0, 40.0],
})
tiny_result = annual_mean_flow_simple(tiny)
print(tiny_result)
assert tiny_result.loc[tiny_result["year"] == 2020, "discharge_cfs"].iloc[0] == 15.0
assert tiny_result.loc[tiny_result["year"] == 2021, "discharge_cfs"].iloc[0] == 40.0
print("Tiny-data test passed.")

In [ ]:
import numpy as np

mine = annual_mean_flow_simple(stream)
package = annual_mean_flow(stream)
check = mine.merge(package, on="year")
assert np.allclose(check["discharge_cfs"], check["mean_discharge_cfs"])
print("My function matches the package function for every year.")

## Guided coding: find the planted bug

The function below contains a deliberate unit bug. Run it, compare against
the package conversion and against the `discharge_m3s` column in the data,
and decide which result is physically believable. A useful anchor to
remember: 1 cfs is about 0.028 m3/s, so 1000 cfs must be about 28 m3/s.

In [ ]:
def cfs_to_m3s_buggy(discharge_cfs):
    """Convert cfs to m3/s. This version contains a bug on purpose."""
    return discharge_cfs * 0.28316846592

from earthcourse.hydro import cfs_to_m3s

print(f"Buggy conversion:   1000 cfs -> {cfs_to_m3s_buggy(1000.0):.1f} m3/s")
print(f"Package conversion: 1000 cfs -> {cfs_to_m3s(1000.0):.1f} m3/s")
print()
print("First rows of the data for comparison:")
print(stream[["discharge_cfs", "discharge_m3s"]].head(3))

The buggy constant is ten times too large. Magnitude checks like this catch
unit errors that no error message will ever report.

## Guided coding: the flow-duration curve

A flow-duration curve shows the percentage of days a given flow is exceeded.
It is a standard summary of a river's behavior, and the package provides it
as a ready-made function.

In [ ]:
import matplotlib.pyplot as plt

from earthcourse.hydro import flow_duration_curve

fdc = flow_duration_curve(stream)
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(fdc["exceedance_probability"] * 100, fdc["discharge_cfs"])
ax.set_xlabel("Percent of days flow is exceeded")
ax.set_ylabel("Discharge (cfs)")
ax.set_title("Flow-duration curve, Iowa River at Iowa City")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Try it yourself

Write a sentence explaining why peak flow should be larger than 7-day low flow in each year.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an
unchanged guided notebook does not meet the submission requirement.

Write `count_days_above(df, threshold_cfs)`. Test it on a three-row DataFrame with a hand-checkable answer, then use it to compare two thresholds in the Iowa River record.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite
one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. Write a function `count_days_above(df, threshold)` that returns the number
   of days with discharge above a threshold. Test it on a three-row
   DataFrame you build yourself, with an `assert`.
2. Use the merged `indices` table to find the year with the lowest 7-day low
   flow. In one sentence, connect that year to drought conditions.
3. Read the flow-duration curve: roughly what flow is exceeded 90 percent of
   the time? What does that number mean for water supply?

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Annual mean flow table
- [ ] 7-day low flow table
- [ ] Annual peak-flow date and value
- [ ] One debugging or sanity check

        ## Short reflection

        What makes a function easier to test than code copied into many cells?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a
valid and useful report.

**Goal:** Replace this text with what you were trying to calculate or show.

**Expected result:** Replace this text.

**What happened:** Replace this text with the result, error, or design choice.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
